In [3]:
import os, random, numpy as np, pandas as pd
from glob import glob
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, balanced_accuracy_score
import timm

# ---------------- 1. Config ----------------
SEED = 42
IMG_SIZE = 299          # Xception was designed for 299x299 images
BATCH = 32
EPOCHS = 15
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

# ---------------- 2. Find the dataset automatically ----------------
def find_dataset(root="/kaggle/input"):
    """Search for HAM10000_metadata.csv anywhere inside /kaggle/input."""
    for folder, _, files in os.walk(root):
        for f in files:
            if f.lower() == "ham10000_metadata.csv":
                return folder, os.path.join(folder, f)
    # Not found: show what IS available so you can debug
    print("Could not find HAM10000_metadata.csv. Contents of", root, ":")
    for item in os.listdir(root):
        print("  -", item)
    raise FileNotFoundError(
        "HAM10000 dataset not attached. Click 'Add Input' in the right sidebar "
        "and add 'Skin Cancer MNIST: HAM10000'."
    )

DATA_DIR, META_PATH = find_dataset()
print("Dataset folder:", DATA_DIR)

# ---------------- 3. Load metadata ----------------
df = pd.read_csv(META_PATH)

# map image_id -> file path (search all sub-folders for .jpg files)
paths = {os.path.splitext(os.path.basename(p))[0]: p
         for p in glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True)}
print("Images found:", len(paths))

df["path"] = df["image_id"].map(paths)
df = df.dropna(subset=["path"]).reset_index(drop=True)
print("Rows with a matching image:", len(df))   # should be 10015

classes = sorted(df["dx"].unique())          # akiec, bcc, bkl, df, mel, nv, vasc
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label"] = df["dx"].map(class_to_idx)
print("Classes:", class_to_idx)

# ---------------- 4. Split BY LESION (avoids data leakage) ----------------
# The same lesion can have several photos. If they land in both train and
# val, the score looks better than it really is. So we split by lesion_id.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["lesion_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["lesion_id"]))
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))

# ---------------- 5. Dataset + augmentation ----------------
mean, std = [0.5]*3, [0.5]*3   # Xception uses 0.5/0.5 normalization

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class HAMDataset(Dataset):
    def __init__(self, frame, tf):
        self.frame, self.tf = frame, tf
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        return self.tf(img), int(row["label"])

train_loader = DataLoader(HAMDataset(train_df, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(HAMDataset(val_df, eval_tf), batch_size=BATCH,
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(HAMDataset(test_df, eval_tf), batch_size=BATCH,
                         shuffle=False, num_workers=2, pin_memory=True)

# ---------------- 6. Model: Xception (pretrained) ----------------
# In newer timm versions the original Xception is called "legacy_xception".
try:
    model = timm.create_model("legacy_xception", pretrained=True, num_classes=len(classes))
except Exception:
    model = timm.create_model("xception", pretrained=True, num_classes=len(classes))
model = model.to(DEVICE)

# ---------------- 7. Loss with class weights ----------------
# 'nv' (moles) is ~67% of the data, so we give rare classes more weight.
counts = train_df["label"].value_counts().sort_index().values
weights = torch.tensor(len(train_df) / (len(classes) * counts), dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)   # mixed precision = faster

# ---------------- 8. Train / evaluate functions ----------------
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, preds_all, labels_all = 0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            preds_all += out.argmax(1).cpu().tolist()
            labels_all += y.cpu().tolist()
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    bal_acc = balanced_accuracy_score(labels_all, preds_all)
    return total_loss / len(loader.dataset), acc, bal_acc

# ---------------- 9. Training loop ----------------
best_bal_acc = 0
for epoch in range(EPOCHS):
    tr_loss, tr_acc, tr_bal = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_bal = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} bal-acc {va_bal:.3f}")
    if va_bal > best_bal_acc:
        best_bal_acc = va_bal
        torch.save(model.state_dict(), "/kaggle/working/xception_ham10000.pth")

# ---------------- 10. Final test ----------------
model.load_state_dict(torch.load("/kaggle/working/xception_ham10000.pth"))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(DEVICE))
        preds += out.argmax(1).cpu().tolist()
        labels += y.tolist()
print(classification_report(labels, preds, target_names=classes, digits=3))

Dataset folder: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
Images found: 10015
Rows with a matching image: 10015
Classes: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
Train / Val / Test: 7991 1025 999
Epoch 1/15 | train loss 1.243 acc 0.620 | val loss 0.907 acc 0.718 bal-acc 0.713
Epoch 2/15 | train loss 0.700 acc 0.736 | val loss 0.817 acc 0.725 bal-acc 0.690
Epoch 3/15 | train loss 0.521 acc 0.773 | val loss 0.744 acc 0.725 bal-acc 0.707
Epoch 4/15 | train loss 0.425 acc 0.803 | val loss 0.711 acc 0.748 bal-acc 0.731
Epoch 5/15 | train loss 0.353 acc 0.825 | val loss 0.608 acc 0.780 bal-acc 0.702
Epoch 6/15 | train loss 0.309 acc 0.851 | val loss 0.696 acc 0.751 bal-acc 0.754
Epoch 7/15 | train loss 0.250 acc 0.869 | val loss 0.553 acc 0.809 bal-acc 0.776
Epoch 8/15 | train loss 0.219 acc 0.883 | val loss 0.640 acc 0.784 bal-acc 0.762
Epoch 9/15 | train loss 0.168 acc 0.899 | val loss 0.660 acc 0.793 bal-acc 0.749
Epoch 10/15 | train loss 0.15